<div style="color: #c00000; border: 2px solid #c00000; padding: 12px; margin-bottom: 16px;">
<strong>WARNING: These files can be large.</strong> The complete course source sample is about <strong>10.75 GB</strong>, and the inclusive collision-data collection is about <strong>1.35 TB</strong>. If you choose to download large samples, consider running this notebook locally on a laptop with plenty of free disk space. Check the storage summary in Section 3 before downloading, and leave extra space for your analysis outputs. The default starter example downloads only about <strong>36.4 MB</strong>.<br><br>
<strong>Using DataHub in your laptop browser still stores files on DataHub.</strong> To use your laptop's disk, run Jupyter and the Python kernel locally. On DataHub, also check your personal storage quota.
</div>

# Downloading original ATLAS Open Data files
### Physics 129 — diphoton events and b-tagged jets

This companion to `physics129_hyy_bjet.ipynb` shows how to **find, download and inspect files directly from the CERN Open Data portal**. All file lists, sizes, checksums and ROOT files come from CERN; no course download script or NERSC mirror is needed.

**Run from top to bottom with Shift+Enter.** The default example downloads two original `GamGam` files: one collision-data period and one $t\bar tH\to\gamma\gamma$ simulation sample, about **36.4 MB** in total. Files go to `~/hyy_original/GamGam`. You can choose more files in Section 3.

**What does “unskimmed” mean here?**

- **Original `GamGam` files:** before the course's requirement of at least one b-tagged jet. ATLAS has already required at least two photons with $p_T\geq25$ GeV.
- **ATLAS `noskim` files:** no additional event skim at the ntuple-making stage. Set `RELEASE = "noskim"` below to use these instead. They still contain reconstructed objects and retain the upstream data-taking and processing selections; they are not raw detector data.

This notebook uses the **2025 education and outreach beta release**, with the tree named `analysis`. Do not mix these files with the older 2020 release, which uses different branch conventions.

*Derived from public ATLAS Open Data. Not an ATLAS Collaboration result.*

## 1. Find the dataset on the portal

Start at [ATLAS Open Data](https://opendata.atlas.cern/) and follow the education and outreach data links. For this notebook, these are the exact CERN records:

| Collection | Collision data | Simulation |
|---|---|---|
| Original diphoton files (`GamGam`) | [93915](https://opendata.cern.ch/record/93915) | [93922](https://opendata.cern.ch/record/93922) |
| Inclusive ntuples (`noskim`) | [93933](https://opendata.cern.ch/record/93933) | [93911](https://opendata.cern.ch/record/93911) |

On a record page, read the selection and then look under **Files and indexes**. You can download an individual ROOT file there. The code below reads the same record as JSON, so you can select several files without copying their long names by hand. Public downloads do not require a CERN login.

The complete `GamGam` collision-data collection is **16 files, 9.86 GB**. The inclusive collision-data collection is about **1.35 TB**. Sizes below use decimal MB/GB. On DataHub, check your personal storage quota before selecting a large collection; filesystem free space is not your personal quota.

In [ ]:
# Install only packages that are missing from the current notebook environment.
import importlib.util
import subprocess
import sys

packages = ["uproot", "awkward", "numpy", "pandas", "matplotlib", "requests", "aiohttp"]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Ready")

## 2. Read the official file lists

A **record ID** identifies a collection on the portal. A **DSID** identifies a simulated physics sample; for example, `346189` is the nominal $t\bar tH$ sample used in the course notebook. The catalog contains many other processes too.

The portal lists `root://` addresses for XRootD clients. Here we use HTTPS to access the same EOS file path, which works with ordinary Python. Preserve the full path: the `noskim` and `GamGam` files live in different directories.

In [ ]:
from pathlib import Path
from urllib.parse import urlsplit, quote
import urllib.request
import json
import re
import pandas as pd
from IPython.display import display

RELEASE = "GamGam"                  # "GamGam" or "noskim"
RECORDS = {"GamGam": {"data": 93915, "MC": 93922},
           "noskim": {"data": 93933, "MC": 93911}}
DEST = Path("~/hyy_original").expanduser() / RELEASE

NOMINAL = {"343981": "ggH", "346214": "VBF", "345317": "W-H", "345318": "W+H",
           "345319": "ZH", "345061": "ggZH", "346189": "ttH",
           "346188": "tHjb", "346486": "tWH"}
ALTERNATIVE = {"346525", "346797", "346878", "346879", "346880", "346881",
               "346882", "346526", "346601", "346602", "345944", "500553",
               "500554", "500555", "506813", "506814", "506811"}

def read_catalog(release):
    rows = []
    for kind, record in RECORDS[release].items():
        with urllib.request.urlopen(f"https://opendata.cern.ch/api/records/{record}",
                                    timeout=60) as response:
            metadata = json.load(response)["metadata"]
        for f in metadata["files"]:
            name = f["key"]
            if not name.endswith(".root"):
                continue
            match = re.search(r"(?:^|_)mc_(\d+)\.", name)
            dsid = match.group(1) if match else ""
            group = ("data" if kind == "data" else
                     "nominal-mc" if dsid in NOMINAL else
                     "alt-mc" if dsid in ALTERNATIVE else "other-mc")
            eos_path = urlsplit(f["uri"]).path.lstrip("/")
            rows.append(dict(kind=kind, group=group, dsid=dsid,
                             mode=NOMINAL.get(dsid, ""), name=name,
                             bytes=int(f["size"]), checksum=f["checksum"],
                             url="https://opendata.cern.ch/" + quote(eos_path, safe="/"),
                             record=record, doi=metadata["doi"]))
    return pd.DataFrame(rows).sort_values(["kind", "name"]).reset_index(drop=True)

catalog = read_catalog(RELEASE)
summary = catalog.groupby("group").agg(files=("name", "size"), bytes=("bytes", "sum"))
summary["GB"] = summary["bytes"] / 1e9
print(f"Official {RELEASE} catalog")
display(summary[["files", "GB"]].round(3))

## 3. Choose files before downloading

Edit the three settings below, then rerun this cell and the cells that follow it.

| `SAMPLE_SET` | What is selected |
|---|---|
| `"starter"` | `GamGam`: data15 periodD + nominal ttH; `noskim`: nominal ttH only (about 34.2 MB) |
| `"nominal-mc"` | Nine nominal Higgs samples from the course notebook |
| `"alt-mc"` | Seventeen alternative Higgs simulation samples |
| `"mc"` | Those 26 Higgs samples |
| `"data"` | All 16 collision-data periods |
| `"all"` | The 42 source files for the course skim: data + those 26 Higgs samples |
| `"catalog"` | Every file in the selected two portal records, including other MC processes |

Use `MATCH = "data15"` for 2015 only, or `MATCH = "346189"` for ttH only. `MAX_FILES = 1` limits the selection to the first matching file. Empty `MATCH` and `MAX_FILES = None` impose no further restriction.

For `GamGam`, the nine nominal Higgs files total about **0.17 GB**; the 42 course source files total about **10.75 GB**. The live catalog above is authoritative. Nominal and alternative samples describe overlapping physics: do not add their yields together.

In [ ]:
SAMPLE_SET = "starter"
MATCH = ""
MAX_FILES = None

sets = {"nominal-mc": {"nominal-mc"}, "alt-mc": {"alt-mc"},
        "mc": {"nominal-mc", "alt-mc"}, "data": {"data"},
        "all": {"data", "nominal-mc", "alt-mc"}}
if SAMPLE_SET == "starter":
    mask = catalog.dsid.eq("346189")
    if RELEASE == "GamGam":
        mask |= catalog.name.str.contains("data15_periodD", regex=False)
elif SAMPLE_SET == "catalog":
    mask = pd.Series(True, index=catalog.index)
elif SAMPLE_SET in sets:
    mask = catalog.group.isin(sets[SAMPLE_SET])
else:
    raise ValueError("Choose a SAMPLE_SET from the table above.")

chosen = catalog[mask & catalog.name.str.contains(MATCH, regex=False)].copy()
if MAX_FILES is not None:
    if not isinstance(MAX_FILES, int) or MAX_FILES < 1:
        raise ValueError("MAX_FILES must be a positive integer or None.")
    chosen = chosen.head(MAX_FILES)
if chosen.empty:
    raise ValueError("No files match. Check SAMPLE_SET and MATCH.")
chosen["MB"] = chosen["bytes"] / 1e6
chosen["path"] = [str(DEST / row.kind / row.name) for row in chosen.itertuples()]
display(chosen[["kind", "dsid", "mode", "MB", "name"]].round(2))
print(f"Selected {len(chosen)} files, {chosen['bytes'].sum() / 1e9:.3f} GB")
print(f"Destination: {DEST}")
print("Example direct download URL:", chosen.iloc[0].url)

### Check disk space before downloading

Run the next cell after selecting your files. It shows the **total capacity, used space and free space** of the filesystem containing `DEST`, alongside the total size of your selected files. It does not download anything.

The free-space estimate below assumes you need to download the entire selection. Files already downloaded may reduce the additional space needed; the download cell checks those files separately. Leave extra room for analysis outputs and temporary files.

**This checks the machine running the notebook's Python kernel.** On DataHub or another shared server, filesystem free space can be much larger than your personal storage quota. This cell cannot determine that quota; check it separately before downloading.


In [ ]:
import shutil
from pathlib import Path

# DEST may not exist yet. Check its nearest existing parent on the same filesystem.
storage_path = Path(DEST).expanduser().resolve()
while not storage_path.exists():
    storage_path = storage_path.parent
usage = shutil.disk_usage(storage_path)
selected_bytes = int(chosen["bytes"].sum())

print(f"Download destination: {DEST}")
print(f"Filesystem checked at: {storage_path}")
print(f"Total disk capacity:              {usage.total / 1e9:10.2f} GB")
print(f"Disk space used:                  {usage.used / 1e9:10.2f} GB")
print(f"Disk space free:                  {usage.free / 1e9:10.2f} GB")
print(f"Total selected file size:         {selected_bytes / 1e9:10.3f} GB")
print(f"Free space if all are downloaded: {(usage.free - selected_bytes) / 1e9:10.2f} GB")
if selected_bytes > usage.free:
    print("WARNING: The full selection exceeds free space. Choose fewer files or another destination.")
else:
    print("The full selection fits in filesystem free space; leave room for analysis outputs.")
print("On DataHub/shared servers, also check your personal quota: it may be smaller than this.")


## 4. Download and check the files

This cell downloads **complete ROOT files with every branch and event in the selected portal files**. No b-jet selection is applied during download.

The helper streams to disk, checks the byte count and CERN's Adler-32 checksum, and then renames the temporary `.part` file. Rerunning skips verified files. Interrupted or failed downloads restart that file from the beginning; they do not resume. An existing file that fails verification is replaced only after a valid download is ready.

`DOWNLOAD_LIMIT_GB` is a guard against an accidental large selection. Increase it deliberately after checking the table and your storage quota. It limits the total selected size, including files already present.

In [ ]:
import shutil
import time
import zlib

DOWNLOAD_LIMIT_GB = 1.0

def verified(path, row):
    path = Path(path)
    if not path.is_file() or path.stat().st_size != row["bytes"]:
        return False
    algorithm, expected = row["checksum"].split(":", 1)
    if algorithm != "adler32":
        raise ValueError(f"Unexpected checksum algorithm: {algorithm}")
    value = 1
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024**2), b""):
            value = zlib.adler32(block, value)
    return f"{value & 0xffffffff:08x}" == expected.lower().zfill(8)

def download_file(row):
    path = Path(row["path"])
    if verified(path, row):
        print("Verified, already present:", path.name, flush=True)
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_suffix(path.suffix + ".part")
    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(row["url"], timeout=120) as response, part.open("wb") as out:
                shutil.copyfileobj(response, out, length=4 * 1024**2)
            if not verified(part, row):
                raise OSError("Downloaded size or checksum does not match the portal.")
            part.replace(path)
            print("Downloaded and verified:", path.name, flush=True)
            return
        except OSError as error:
            print(f"Attempt {attempt} failed: {error}", flush=True)
            if attempt == 3:
                raise
            time.sleep(2)

selected_bytes = int(chosen["bytes"].sum())
if selected_bytes > DOWNLOAD_LIMIT_GB * 1e9:
    raise RuntimeError("Selection exceeds DOWNLOAD_LIMIT_GB. Choose fewer files or raise the limit.")
DEST.mkdir(parents=True, exist_ok=True)
needed = sum(row["bytes"] for row in chosen.to_dict("records")
             if not verified(row["path"], row))
if needed > shutil.disk_usage(DEST).free:
    raise OSError("Not enough filesystem space. Choose fewer files or change DEST.")

for row in chosen.to_dict("records"):
    print(f"Checking {row['name']} ({row['bytes'] / 1e6:.1f} MB)", flush=True)
    download_file(row)

# Save the selected source URLs, record IDs, sizes and checksums for your analysis notes.
manifest_path = DEST / "last_download_manifest.json"
manifest_path.write_text(chosen.drop(columns=["MB", "path"]).to_json(orient="records", indent=2))
print(f"Done. Source information saved in {manifest_path}")

## 5. Inspect a ROOT file

`uproot` reads ROOT files without installing the ROOT application. A tree is an event table; branches hold event quantities or variable-length lists of reconstructed particles. The downloaded files stay unchanged when we read a subset of branches or entries.

In [ ]:
import uproot
import awkward as ak
import numpy as np

inventory = []
for row in chosen.to_dict("records"):
    with uproot.open(row["path"]) as root_file:
        tree = root_file["analysis"]
        inventory.append(dict(kind=row["kind"], dsid=row["dsid"],
                              entries=tree.num_entries, branches=len(tree.keys()), name=row["name"]))
display(pd.DataFrame(inventory))

example_path = chosen.iloc[0].path
with uproot.open(example_path) as root_file:
    tree = root_file["analysis"]
    print("Trees/objects:", root_file.keys())
    print("Photon branches:", [b for b in tree.keys() if b.startswith("photon_")])
    events = tree.arrays(["photon_pt", "photon_eta", "jet_btag_quantile"], entry_stop=5)
print("First five events:")
print(ak.to_list(events))

## 6. First look: apply the b-jet requirement yourself

Process **all entries of the first selected file**, in small batches. Count events before and after `jet_btag_quantile >= 4` (the 70% efficiency working point used in the course notebook), and compare diphoton masses.

For the plot, require two tight, isolated photons with $p_T>25$ GeV, $|\eta|<2.37$, excluding $1.37<|\eta|<1.52$. Use the two highest-$p_T$ accepted photons, require $p_{T,1}/m_{\gamma\gamma}>0.35$ and $p_{T,2}/m_{\gamma\gamma}>0.25$, and plot $105<m_{\gamma\gamma}<160$ GeV. Momenta and energies in these ntuples are in GeV.

The plot shows **unweighted entries in one file**, including for MC. It illustrates the cut and is not an expected signal yield or a comparison normalized to 36.1 fb⁻¹. The first selected file is identified in the output.

In [ ]:
import matplotlib.pyplot as plt

branches = ["photon_pt", "photon_eta", "photon_phi", "photon_e",
            "photon_isTightID", "photon_isTightIso", "jet_btag_quantile"]
bins = np.arange(105, 160.1, 2.5)
before = np.zeros(len(bins) - 1, dtype=int)
after = np.zeros_like(before)
n_all = n_bjet = 0

with uproot.open(example_path) as root_file:
    for ev in root_file["analysis"].iterate(branches, step_size="25 MB", library="ak"):
        has_bjet = ak.any(ev.jet_btag_quantile >= 4, axis=1)
        n_all += len(ev)
        n_bjet += int(ak.sum(has_bjet))
        aeta = abs(ev.photon_eta)
        good = ((ev.photon_pt > 25) & (aeta < 2.37)
                & ~((aeta > 1.37) & (aeta < 1.52))
                & ev.photon_isTightID & ev.photon_isTightIso)
        order = ak.argsort(ev.photon_pt[good], ascending=False)
        photons = {k: ev[f"photon_{k}"][good][order] for k in ("pt", "eta", "phi", "e")}
        two = ak.num(photons["pt"]) >= 2
        if not ak.any(two):
            continue
        pt, eta, phi, energy = (ak.to_numpy(photons[k][two][:, :2])
                                for k in ("pt", "eta", "phi", "e"))
        px = (pt * np.cos(phi)).sum(axis=1)
        py = (pt * np.sin(phi)).sum(axis=1)
        pz = (pt * np.sinh(eta)).sum(axis=1)
        mass = np.sqrt(np.maximum(energy.sum(axis=1)**2 - px**2 - py**2 - pz**2, 0))
        keep = ((mass > 105) & (mass < 160)
                & (pt[:, 0] > 0.35 * mass) & (pt[:, 1] > 0.25 * mass))
        before += np.histogram(mass[keep], bins=bins)[0]
        after += np.histogram(mass[keep & ak.to_numpy(has_bjet[two])], bins=bins)[0]

print("File:", Path(example_path).name)
print(f"All entries: {n_all:,}; at least one b-tagged jet: {n_bjet:,}")
if n_all:
    print(f"B-jet acceptance before photon analysis cuts: {100 * n_bjet / n_all:.2f}%")
print(f"Diphoton selection: {before.sum():,}; also requiring a b-jet: {after.sum():,}")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.stairs(before, bins, label="Diphoton selection", linewidth=1.8)
ax.stairs(after, bins, label="Also at least one b-tagged jet", linewidth=1.8)
ax.set(xlabel=r"$m_{\gamma\gamma}$ [GeV]", ylabel="Unweighted entries / 2.5 GeV",
       title=f"ATLAS Open Data — one {RELEASE} file")
ax.legend()
fig.tight_layout()
plt.show()

## 7. Read a small sample remotely instead (optional)

If disk space is limited, `uproot` can read an HTTPS URL directly. It transfers the needed compressed baskets, so reading 1,000 entries can still transfer more than exactly those entries. No full local ROOT copy is saved by this cell. Repeated remote scans can be slower than downloading once.

This example is off by default. To try it, set `READ_REMOTE = True`. You need only Sections 1–3, so you can skip the download and local analysis cells entirely.

In [ ]:
import uproot
import awkward as ak

READ_REMOTE = False
if READ_REMOTE:
    remote_url = chosen.iloc[0].url
    with uproot.open(remote_url, timeout=120) as root_file:
        sample = root_file["analysis"].arrays(
            ["photon_pt", "jet_btag_quantile"], entry_stop=1000, library="ak")
    print(f"Read {len(sample):,} events remotely")
    print(ak.to_list(sample[:3]))
else:
    print("Optional remote read skipped.")

## 8. Use the inclusive `noskim` files or another process

To remove the ATLAS diphoton skim as well, set `RELEASE = "noskim"` in Section 2 and rerun from there. Keep `SAMPLE_SET = "starter"` for a small **ttH-only** example. The same download, checksum and analysis code works; the two-photon requirement is then imposed by your analysis.

For inclusive collision data, choose `SAMPLE_SET = "data"` and a narrow `MATCH`, then inspect the sizes. Even `data15_periodD.noskim.root` is about **9.45 GB**. The default 1 GB limit will stop that download. Use remote reads for exploration, or choose a machine with sufficient disk and memory resources before increasing the limit.

For a different simulated process, inspect the catalog and choose its DSID with `SAMPLE_SET = "catalog"`, `MATCH = "<DSID>"`. Inspect its generator description and normalization before using it. A filename filter selects files; it does not filter events inside them.

In [ ]:
# Search the official catalog without downloading anything. Edit the literal text.
SEARCH_TEXT = "ttH"
view = catalog[catalog.name.str.contains(SEARCH_TEXT, case=False, regex=False)].copy()
view["MB"] = view["bytes"] / 1e6
display(view[["dsid", "group", "MB", "name"]].round(2))

## Good to know

- **Luminosity:** the complete 2015+2016 data collection corresponds to 36.1 fb⁻¹. A single period or a truncated read is only a subset; do not label it as the full dataset.
- **MC weights:** the course example uses `mcWeight * xsec * kfac * filteff * LUMI_PB / sum_of_weights` for its nominal samples. Set `LUMI_PB` to the luminosity you intend to predict. Retain the provided full-sample normalization after event cuts; do not replace `sum_of_weights` with the sum after your selection. A first-N-entry test is not automatically a full-sample prediction. For other samples, check cross sections, branching ratios and relevant `ScaleFactor_*` branches before quoting yields.
- **Memory:** `tree.arrays()` without an entry limit can load a large dataset. Use selected branches and `tree.iterate()` for full scans, as above. A compressed file's size is smaller than its memory footprint after reading.
- **Connection or quota errors:** retry the cell after restoring connectivity or choosing a smaller selection. A `.part` file is an incomplete download; this notebook restarts it. Do not disable certificate verification to work around HTTPS errors.
- **Next exercise:** compare the b-jet acceptance of ggH (`343981`) and ttH (`346189`). Select and download each sample, then rerun the first-look cells. Explain the difference using the particles produced with the Higgs boson.

### Sources and acknowledgements

Use the [ATLAS 2025 education and outreach release](https://opendata.cern.ch/record/93910) and cite the particular records you used: [GamGam data](https://opendata.cern.ch/record/93915), [GamGam MC](https://opendata.cern.ch/record/93922), [noskim data](https://opendata.cern.ch/record/93933), [noskim MC](https://opendata.cern.ch/record/93911). Record IDs and DOIs are also saved in the download manifest. See the [ATLAS ntuple documentation](https://opendata.atlas.cern/docs/data/for_education/13TeV25_details) for branch definitions and selections. These data are released under CC0; acknowledge ATLAS and follow the citation guidance on the records.